# 05.01 · Provider landscape & choosing models

**Chapter:** Working with LLM APIs · **Milestone:** M2

## Goals
- [ ] Map the three kinds of providers: closed frontier, hosted open-weight, local
- [ ] Compare models on the dimensions that matter: quality, price, latency (TTFT, tokens/s), context, data policy
- [ ] Measure latency yourself instead of trusting marketing numbers

## Resources
- [Artificial Analysis](https://artificialanalysis.ai/) — independent quality / speed / price comparisons — free
- [OpenRouter models](https://openrouter.ai/models) — one catalogue with prices for hundreds of models — free
- [LMArena](https://lmarena.ai/) — crowd-sourced head-to-head rankings — free

Setup: this chapter uses the repo helper `aieng.llm` (one OpenAI-compatible client for every provider, built from 00.03).
Everything runs on local Ollama (`ollama pull qwen3:1.7b`) with zero keys; add free Gemini / Groq keys to `.env` to compare.

## 1. Three kinds of providers

| Kind | Examples | Pros | Cons |
|---|---|---|---|
| **Closed frontier APIs** | OpenAI (GPT), Anthropic (Claude), Google (Gemini), xAI (Grok), Mistral | best quality, multimodal, tools, huge context | paid (some free tiers), data leaves your infra, model can change/deprecate |
| **Hosted open-weight** | Groq, Cerebras, Together, Fireworks, DeepInfra, OpenRouter (aggregator) | cheap, very fast, swap models freely, same weights you could self-host | quality below frontier on hardest tasks |
| **Local / self-hosted** | Ollama, LM Studio, llama.cpp, MLX, vLLM (GPU) | free per call, private, offline, full control | your hardware limits size/speed; you operate it |

Most production systems mix them: a frontier model for hard steps, a small/cheap one for easy steps (routing, chapter 11).

## 2. What to compare
- **Quality on *your* task** — public benchmarks are a starting filter; your own eval set (chapter 10) decides.
- **Price** — per 1M input and output tokens (output usually 3–5× pricier). Cached-input discounts, batch discounts.
- **Latency** — *time to first token* (TTFT, what users feel) and *output tokens/second* (how long answers take).
- **Context window** and max output tokens.
- **Features** — tool calling, structured outputs, vision, audio, prompt caching, batch API.
- **Data policy** — free tiers may train on your data; check retention, region, compliance needs.
- **Rate limits** — requests/tokens per minute and per day.

In [ ]:
import time

from aieng.llm import PROVIDERS, available, client

print("configured providers:", available())

## 3. Measure TTFT and tokens/second
Stream the response; the first content chunk marks TTFT. Ask for usage stats so we know the output token count.

In [ ]:
def measure(provider, prompt="Explain how a hash map works in about 150 words.", **kwargs):
    c, model = client(provider)
    if c.base_url.host == "localhost":
        kwargs.setdefault("reasoning_effort", "none")
    start = time.perf_counter()
    ttft, text, usage = None, "", None
    stream = c.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}], stream=True,
                                       stream_options={"include_usage": True}, **kwargs)
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            ttft = ttft or time.perf_counter() - start
            text += chunk.choices[0].delta.content
        if chunk.usage:
            usage = chunk.usage
    total = time.perf_counter() - start
    out_tokens = usage.completion_tokens if usage else len(text) / 4
    return {"provider": provider, "model": model, "ttft_s": round(ttft, 2), "total_s": round(total, 2),
            "out_tokens": out_tokens, "tok_per_s": round(out_tokens / max(total - ttft, 1e-6))}


results = []
for p in available():
    try:
        results.append(measure(p))
    except Exception as e:  # missing model, provider down, rate limit...
        print(f"{p}: {type(e).__name__}: {str(e)[:120]}")

import pandas as pd
pd.DataFrame(results)

Run it several times: latency varies a lot by time of day and load. Report medians, never a single run.

## 4. Price math
Cost per request = `input_tokens × in_price + output_tokens × out_price` (prices per 1M tokens).
Below: **illustrative** prices — fill in current ones from each provider's pricing page or openrouter.ai/models.

In [ ]:
illustrative_prices = {  # $ per 1M tokens (input, output). UPDATE THESE from real pricing pages.
    "small fast model": (0.10, 0.40),
    "mid-tier model": (1.00, 4.00),
    "frontier model": (3.00, 15.00),
    "local (electricity only)": (0.0, 0.0),
}

def monthly_cost(in_tok, out_tok, requests_per_day, price):
    per_request = (in_tok * price[0] + out_tok * price[1]) / 1e6
    return per_request, per_request * requests_per_day * 30

# A RAG chatbot: ~3,000 input tokens (question + retrieved context), ~300 output, 5,000 requests/day
for name, price in illustrative_prices.items():
    per_req, month = monthly_cost(3000, 300, 5000, price)
    print(f"{name:26} ${per_req:.5f}/request -> ${month:,.0f}/month")

Notice input tokens dominate for RAG/agents (big contexts, short answers). That's why prompt caching and context
engineering (06.04) matter so much for cost.

## Exercises
1. Get free keys for at least two of Gemini / Groq / OpenRouter and rerun `measure` 5× per provider. Report median TTFT and tokens/s.
2. Build a table of 6 models (from Artificial Analysis / OpenRouter): $/1M in & out, context window, tokens/s, one quality index.
   Pick one model per use case: (a) customer-support chatbot, (b) offline batch summarisation of 1M documents, (c) coding agent.
3. Recompute the monthly cost with real prices. At what request volume does renting a GPU for an open model become cheaper?

In [ ]:
# your code here